# Capstone — Content Archetype Clustering for Editorial Triage

## Abstract

This capstone uses leakage-audited, rolling-90-day telemetry to describe recurring content-performance patterns. It fits K-Means clusters, evaluates their geometry and client-held-out coherence, assigns human-readable archetype names after profile inspection, and supplies a transparent human-review queue. The work is decision-support, not causal evidence that a content intervention will change search outcomes.

## 1. Question and decision

**Question:** What descriptive content-performance archetypes appear in the active corpus when reach, ranking, click efficiency, freshness, and engagement are considered together?

**Decision supported:** Which pages should an editor review first, and what contextual checks should accompany that review?

## 2. Data and scope

The source is the anonymized starter dataset, with a rolling 90-day measurement window. The active-corpus contract requires at least 10 impressions and 90 days of content age. Pages with `avg_position == 0` are removed from this model because zero means no ranking data rather than rank zero. Identifiers, URLs, raw queries, target-derived fields, and product output fields are not model features.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
contract_mask = (df_raw['impressions_90d'] >= 10) & (df_raw['content_age_days'] >= 90)
df_clean = df_raw.loc[contract_mask & (df_raw['avg_position'] > 0)].copy()
X = pd.DataFrame({
    'impressions_log': np.log1p(df_clean['impressions_90d']),
    'avg_position': df_clean['avg_position'],
    'ctr': df_clean['ctr'],
    'staleness_log': np.log1p(df_clean['days_since_last_update']),
    'engagement_rate': df_clean['engagement_rate'],
})
X_scaled = StandardScaler().fit_transform(X)
print(f'Model corpus: {len(df_clean):,} pages across {df_clean["client_id"].nunique()} pseudonymized clients')


## 3. Methodology and feature selection

The leakage-safe candidate vector had 11 variables. The final five features were selected to represent distinct objective dimensions: `impressions_log` (reach), `avg_position` (ranking), `ctr` (click efficiency), `staleness_log` (freshness), and `engagement_rate` (user behavior). Clicks and sessions overlap strongly with impressions; update ratio overlaps with staleness; word count has meaningful missingness and is not a behavior signal; scroll rate is an additional engagement signal; content age is less direct than staleness for the triage objective.

K-Means uses Euclidean distance, so heavy-tailed reach and freshness are transformed with `log1p` and all five final dimensions are standardized.

### Select K and check stability

`K=5` is not chosen because five personas would be convenient. Values from 2 to 8 are compared using inertia, silhouette, Davies–Bouldin, cluster sizes, seed stability, and interpretability.

In [ ]:
sample_idx = np.arange(0, len(X_scaled), 10)
k_rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    k_rows.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(X_scaled[sample_idx], labels[sample_idx]),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels),
        'smallest_cluster': pd.Series(labels).value_counts().min(),
    })
k_results = pd.DataFrame(k_rows)
display(k_results.round(3))

final_k = 5
base_labels = KMeans(n_clusters=final_k, random_state=0, n_init=20).fit_predict(X_scaled)
stability_rows = []
for seed in [42, 100, 2026]:
    labels = KMeans(n_clusters=final_k, random_state=seed, n_init=20).fit_predict(X_scaled)
    stability_rows.append({
        'seed': seed,
        'silhouette': silhouette_score(X_scaled[sample_idx], labels[sample_idx]),
        'ARI_vs_seed_0': adjusted_rand_score(base_labels, labels),
    })
display(pd.DataFrame(stability_rows).round(3))


## 4. Results and held-out client coherence

For this extract, the comparison supports retaining `K=5`. To evaluate transfer beyond fit clients, the analysis fits centroids on four client groups and assigns pages from the held-out client group. The resulting silhouette is a held-out coherence measure, not classification accuracy.

In [ ]:
fold_rows = []
gkf = GroupKFold(n_splits=5)
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=df_clean['client_id']), start=1):
    fold_scaler = StandardScaler()
    X_train = fold_scaler.fit_transform(X.iloc[train_idx])
    X_test = fold_scaler.transform(X.iloc[test_idx])
    fold_model = KMeans(n_clusters=final_k, random_state=42, n_init=20).fit(X_train)
    test_labels = fold_model.predict(X_test)
    test_sample = np.arange(0, len(X_test), 10)
    fold_rows.append({
        'fold': fold,
        'held_out_clients': df_clean.iloc[test_idx]['client_id'].nunique(),
        'held_out_pages': len(test_idx),
        'held_out_silhouette': silhouette_score(X_test[test_sample], test_labels[test_sample]),
    })
holdout_results = pd.DataFrame(fold_rows)
display(holdout_results.round(3))

kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=20)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)
cluster_names = {
    0: 'Stale, High-Reach',
    1: 'Current, High-Reach',
    2: 'High-Engagement',
    3: 'Low-Visibility',
    4: 'High-CTR, Low-Reach',
}
df_clean['archetype'] = df_clean['cluster'].map(cluster_names)
profiles = df_clean.groupby('archetype').agg(
    pages=('content_id', 'size'),
    share_of_corpus=('content_id', lambda s: len(s) / len(df_clean) * 100),
    median_impressions=('impressions_90d', 'median'),
    median_position=('avg_position', 'median'),
    median_ctr=('ctr', 'median'),
    median_staleness=('days_since_last_update', 'median'),
    median_engagement=('engagement_rate', 'median'),
).round(2)
display(profiles)


### Cluster naming

K-Means produces arbitrary numeric cluster IDs. The archetype names are semantic labels assigned by a human after examining the profile table; they are not direct model outputs.

## 5. Recommendations and heuristic scoring

Pages are routed to review playbooks: stale high-reach pages receive factual/content-refresh review; current high-reach pages are monitored; high-engagement pages are examined for transferable experience patterns; low-visibility pages receive intent/link/consolidation review; and high-CTR low-reach pages receive query-coverage and internal-linking review.

The ranking score used in the action playbook is a manual heuristic: 40% reach, 35% ranking opportunity, and 25% staleness. It is not trained on intervention outcomes. The playbook tests reasonable alternative weights and requires human review of intent, redirects, backlinks, factual accuracy, and YMYL risk before action.

## 6. Decision log

| Decision | Evidence | Result |
| --- | --- | --- |
| Use log transforms | Heavy right-skew in counts | Accepted |
| Standardize dimensions | K-Means distance is scale-sensitive | Accepted |
| Exclude leakage/product fields | Decision-time, privacy, and circularity audit | Accepted |
| Retain five features | Redundancy, relevance, and missingness review | Accepted |
| Retain `K=5` | K comparison, sizes, stability, and interpretation | Accepted for this extract |
| Use client-held-out check | Prevent same-client fit/holdout overlap | Accepted |
| Name clusters | Human reading of profile distributions | Post-hoc interpretation |

## 7. Limitations and skeptic check

- Clustering is descriptive; it does not establish causes or intervention effects.
- `K` and the feature definitions are modelling choices; results may change when either changes.
- Cluster names are human interpretations, not model labels.
- The action score is a heuristic, not a causal or supervised score.
- Client distribution and unobserved search-environment changes can affect the observed structure.
- A controlled intervention design would be needed to estimate whether any recommended action improves an outcome.

## Acknowledgments and data credit

This research was developed as part of the **FlyRank Applied Machine Learning Internship**.

*Data credit:* Built on the [FlyRank](https://flyrank.ai) ML Internship dataset.

## Self-check

- [x] All quantitative results are calculated from the current code.
- [x] The capstone distinguishes associations, cluster coherence, and causal claims.
- [x] Cluster IDs and human archetype names are separated.
- [x] No client names, URLs, or raw queries are displayed.